The following is the code needed to download the data we used in this assignment. <br>
To run it, log into WRDS, then go to PROGRAMS > JupyterHub > Open JupyterHub > New Launcher and make a notebook with a python 3 kernal. <br>
Finally, paste the below code in, and it should run. You may need to change the username in the line "db = wrds.Connection(wrds_username="nzanatta")"

In [17]:

import wrds
import pandas as pd
import numpy as np
import csv
from pathlib import Path
from IPython.display import display, FileLink

# ==========================================
# 1. SETTINGS
# ==========================================

END_DATE = "2025-12-31"

OUTPUT_DIR = Path.cwd()
SHILLER_FILE = OUTPUT_DIR / "ie_data.csv"
OUTPUT_FILE = OUTPUT_DIR / "market_data_full_history_2025.csv"

WRDS_USERNAME = "nzanatta"

print(f"Working directory: {OUTPUT_DIR}")
print("Start date: Earliest available")
print(f"End date: {END_DATE}")

# ==========================================
# 2. READ SHILLER CSV
# ==========================================

if not SHILLER_FILE.exists():
    raise FileNotFoundError(
        f"Cannot find {SHILLER_FILE}. "
        "Upload ie_data.csv to your JupyterHub folder."
    )

print("\nLoading Shiller data...")

with open(
    SHILLER_FILE,
    "r",
    encoding="utf-8-sig",
    newline=""
) as f:
    rows = list(csv.reader(f))

# Locate table header
header_index = None

for i, row in enumerate(rows):

    values = [
        str(x).strip().upper()
        for x in row
    ]

    if all(x in values for x in ["DATE", "P", "D", "E"]):

        header_index = i

        positions = [
            values.index("DATE"),
            values.index("P"),
            values.index("D"),
            values.index("E")
        ]

        break

if header_index is None:
    raise ValueError(
        "Could not find Date, P, D, E columns. "
        "Make sure you saved the Data worksheet as CSV."
    )

# Extract relevant columns
records = []

for row in rows[header_index + 1:]:

    if len(row) <= max(positions):
        continue

    records.append([row[p] for p in positions])

shiller_df = pd.DataFrame(
    records,
    columns=[
        "year_month",
        "sp500_price",
        "sp500_dividends",
        "sp500_earnings"
    ]
)

# Convert data to numeric
for col in shiller_df.columns:
    shiller_df = shiller_df.assign(
        **{
            col: pd.to_numeric(
                shiller_df[col],
                errors="coerce"
            )
        }
    )

shiller_df = shiller_df.dropna(
    subset=["year_month"]
).copy()

# ==========================================
# 3. CONVERT SHILLER DATES
# ==========================================

# Shiller uses YYYY.MM format:
# 1871.01 = January 1871
# 1871.12 = December 1871

year = np.floor(
    shiller_df["year_month"]
).astype(int)

month = np.rint(
    (shiller_df["year_month"] - year) * 100
).astype(int)

valid = (
    month.between(1, 12)
    & year.between(1800, 2100)
)

shiller_df = shiller_df.loc[valid].copy()

year = year.loc[valid]
month = month.loc[valid]

dates = pd.to_datetime({
    "year": year.to_numpy(),
    "month": month.to_numpy(),
    "day": np.ones(len(year), dtype=int)
})

shiller_df = shiller_df.assign(
    date=dates + pd.offsets.MonthEnd(0)
)

# ==========================================
# 4. CALCULATE DIVIDEND AND EARNINGS YIELDS
# ==========================================

# Shiller D and E represent annual amounts.
#
# Dividend yield = Annual dividends / Price
# Earnings yield = Annual earnings / Price
#
# Both yields are annual-basis ratios,
# observed at monthly frequency.

price = shiller_df["sp500_price"].where(
    shiller_df["sp500_price"] > 0
)

shiller_df = shiller_df.assign(
    dividend_yield=(
        shiller_df["sp500_dividends"] / price
    ),
    earnings_yield=(
        shiller_df["sp500_earnings"] / price
    )
)

shiller_df = shiller_df[
    [
        "date",
        "sp500_price",
        "sp500_dividends",
        "sp500_earnings",
        "dividend_yield",
        "earnings_yield"
    ]
].copy()

# Keep earliest available data through END_DATE
shiller_df = shiller_df.loc[
    shiller_df["date"] <= pd.Timestamp(END_DATE)
].copy()

shiller_df = (
    shiller_df
    .drop_duplicates(subset=["date"])
    .sort_values("date")
    .reset_index(drop=True)
)

print(f"\nShiller observations: {len(shiller_df):,}")

if not shiller_df.empty:
    print(
        f"Shiller date range: "
        f"{shiller_df['date'].min().date()} "
        f"to {shiller_df['date'].max().date()}"
    )

display(shiller_df.head(10))

# ==========================================
# 5. CONNECT TO WRDS
# ==========================================

print("\nConnecting to WRDS...")

db = wrds.Connection(
    wrds_username=WRDS_USERNAME
)

try:

    # ======================================
    # CRSP MARKET TOTAL RETURNS
    # ======================================

    print("\nDownloading CRSP market returns...")

    market_df = db.raw_sql(f"""
        SELECT
            mthcaldt AS date,
            vwretd AS market_total_return

        FROM crsp_a_stock.wrds_monthlyindexret_query

        WHERE mthcaldt <= DATE '{END_DATE}'

        ORDER BY mthcaldt

    """, date_cols=["date"])

    # ======================================
    # FAMA-FRENCH RISK-FREE RATE
    # ======================================

    print("Downloading Fama-French risk-free rates...")

    rf_df = db.raw_sql(f"""
        SELECT
            dateff AS date,
            rf AS risk_free_rate

        FROM ff_all.factors_monthly

        WHERE dateff <= DATE '{END_DATE}'

        ORDER BY dateff

    """, date_cols=["date"])

finally:
    db.close()

# ==========================================
# 6. CLEAN WRDS DATA
# ==========================================

def clean_dates(df, name):

    if df.empty:
        raise ValueError(
            f"WRDS returned no observations for {name}."
        )

    df = df.copy()

    dates = pd.to_datetime(
        df["date"],
        errors="coerce"
    )

    df = df.assign(
        date=dates.dt.to_period("M").dt.to_timestamp("M")
    )

    df = df.dropna(subset=["date"]).copy()

    duplicates = df.duplicated(
        subset=["date"]
    ).sum()

    if duplicates > 0:
        print(
            f"WARNING: {name} has {duplicates} "
            f"duplicate dates."
        )

    return (
        df.drop_duplicates(subset=["date"])
        .sort_values("date")
        .reset_index(drop=True)
    )

market_df = clean_dates(
    market_df,
    "Market returns"
)

rf_df = clean_dates(
    rf_df,
    "Risk-free rate"
)

market_df = market_df.assign(
    market_total_return=pd.to_numeric(
        market_df["market_total_return"],
        errors="coerce"
    )
)

rf_df = rf_df.assign(
    risk_free_rate=pd.to_numeric(
        rf_df["risk_free_rate"],
        errors="coerce"
    )
)

# Both CRSP and WRDS Fama-French returns
# are treated as decimal monthly returns.
# Example: 0.005 = 0.5% for one month.

# ==========================================
# 7. MERGE ALL DATA
# ==========================================

# Outer merge preserves each source's full
# historical coverage, including dates
# before CRSP market returns begin.

df = shiller_df.merge(
    market_df,
    on="date",
    how="outer",
    validate="one_to_one"
)

df = df.merge(
    rf_df,
    on="date",
    how="outer",
    validate="one_to_one"
)

# Restrict to December 2025
df = df.loc[
    df["date"] <= pd.Timestamp(END_DATE)
].copy()

df = (
    df.sort_values("date")
    .reset_index(drop=True)
)

# ==========================================
# 8. CALCULATE MARKET EXCESS RETURNS
# ==========================================

# Monthly excess market return:
# Total market return - Monthly risk-free return

df = df.assign(
    market_excess_return=(
        df["market_total_return"]
        - df["risk_free_rate"]
    )
)

# ==========================================
# 9. RENAME COLUMNS
# ==========================================

df = df.rename(columns={

    "market_total_return":
        "market_total_return (including dividends)",

    "risk_free_rate":
        "risk_free_rate (1 month T-bill)",

    "dividend_yield":
        "dividend_yield (annual)",

    "earnings_yield":
        "earnings_yield (annual)",

    "sp500_dividends":
        "sp500_dividends (annual)",

    "sp500_earnings":
        "sp500_earnings (annual)"

})

# ==========================================
# 10. FINAL COLUMN ORDER
# ==========================================

df = df[
    [
        "date",
        "market_total_return (including dividends)",
        "risk_free_rate (1 month T-bill)",
        "market_excess_return",
        "dividend_yield (annual)",
        "earnings_yield (annual)",
        "sp500_price",
        "sp500_dividends (annual)",
        "sp500_earnings (annual)"
    ]
].copy()

# ==========================================
# 11. PRINT DATA AVAILABILITY
# ==========================================

print("\n========== DATA AVAILABILITY ==========")

summary = []

for col in df.columns:

    if col == "date":
        continue

    available = df.loc[
        df[col].notna(),
        ["date", col]
    ]

    if available.empty:

        first_date = "MISSING"
        last_date = "MISSING"

    else:

        first_date = (
            available["date"].min().strftime("%Y-%m-%d")
        )

        last_date = (
            available["date"].max().strftime("%Y-%m-%d")
        )

    summary.append({
        "Variable": col,
        "First Date": first_date,
        "Last Date": last_date,
        "Observations": len(available),
        "Missing": df[col].isna().sum()
    })

summary_df = pd.DataFrame(summary)

display(summary_df)

# ==========================================
# 12. SAVE CSV
# ==========================================

df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n========== FILE SAVED ==========")

print(f"Saved: {OUTPUT_FILE}")
print(f"Rows: {len(df):,}")

display(FileLink(str(OUTPUT_FILE)))

# ==========================================
# 13. DISPLAY DATA PREVIEWS
# ==========================================

print("\n========== FIRST 10 ROWS ==========")
display(df.head(10))

print("\n========== LAST 10 ROWS ==========")
display(df.tail(10))

# ==========================================
# 14. SUMMARY STATISTICS
# ==========================================

print("\n========== SUMMARY STATISTICS ==========")

display(df.describe())

# ==========================================
# 15. FULL SAMPLE SUMMARY
# ==========================================

print("\n========== FULL SAMPLE SUMMARY ==========")

print(
    f"First date overall: "
    f"{df['date'].min().date()}"
)

print(
    f"Last date overall: "
    f"{df['date'].max().date()}"
)

# Determine period where all four
# main variables are available.

required_columns = [
    "market_total_return (including dividends)",
    "risk_free_rate (1 month T-bill)",
    "dividend_yield (annual)",
    "earnings_yield (annual)"
]

complete = df.dropna(
    subset=required_columns
).copy()

print(
    f"\nMonths with all four variables: "
    f"{len(complete):,}"
)

if not complete.empty:

    print(
        f"Complete sample range: "
        f"{complete['date'].min().date()} "
        f"to {complete['date'].max().date()}"
    )

# ==========================================
# 16. FINAL COLUMN EXPLANATIONS
# ==========================================

print("\n========== VARIABLE DEFINITIONS ==========")

definitions = {
    "market_total_return (including dividends)":
        "Monthly CRSP value-weighted market total return",

    "risk_free_rate (1 month T-bill)":
        "Monthly return on 1-month Treasury bills",

    "market_excess_return":
        "Monthly market return minus risk-free return",

    "dividend_yield (annual)":
        "Annual dividends divided by S&P composite price",

    "earnings_yield (annual)":
        "Annual earnings divided by S&P composite price",

    "sp500_price":
        "Shiller S&P composite monthly price",

    "sp500_dividends (annual)":
        "Annual dividends per index unit",

    "sp500_earnings (annual)":
        "Annual earnings per index unit"
}

definitions_df = pd.DataFrame(
    list(definitions.items()),
    columns=["Variable", "Definition"]
)

display(definitions_df)

print("\n========== COMPLETE ==========")
print(f"Output directory: {OUTPUT_DIR}")


Working directory: /home/uwaterloo/nzanatta/A3
Start date: Earliest available
End date: 2025-12-31

Loading Shiller data...

Shiller observations: 1,860
Shiller date range: 1871-01-31 to 2025-12-31


,date,sp500_price,sp500_dividends,sp500_earnings,dividend_yield,earnings_yield
0,1871-01-31,4.44,0.26,0.4,0.058559,0.090090
1,1871-02-28,4.50,0.26,0.4,0.057778,0.088889
2,1871-03-31,4.61,0.26,0.4,0.056399,0.086768
3,1871-04-30,4.74,0.26,0.4,0.054852,0.084388
4,1871-05-31,4.86,0.26,0.4,0.053498,0.082305
5,1871-06-30,4.82,0.26,0.4,0.053942,0.082988
6,1871-07-31,4.73,0.26,0.4,0.054968,0.084567
7,1871-08-31,4.79,0.26,0.4,0.054280,0.083507
8,1871-09-30,4.84,0.26,0.4,0.053719,0.082645
9,1871-10-31,4.59,0.26,0.4,0.056645,0.087146



Connecting to WRDS...
Loading library list...
Done


========== DATA AVAILABILITY ==========


,Variable,First Date,Last Date,Observations,Missing
0,market_total_return (including dividends),1926-01-31,2025-12-31,1200,660
1,risk_free_rate (1 month T-bill),1926-07-31,2025-12-31,1194,666
2,market_excess_return,1926-07-31,2025-12-31,1194,666
3,dividend_yield (annual),1871-01-31,2025-12-31,1860,0
4,earnings_yield (annual),1871-01-31,2025-12-31,1860,0
5,sp500_price,1871-01-31,2025-12-31,1860,0
6,sp500_dividends (annual),1871-01-31,2025-12-31,1860,0
7,sp500_earnings (annual),1871-01-31,2025-12-31,1860,0



========== FILE SAVED ==========
Saved: /home/uwaterloo/nzanatta/A3/market_data_full_history_2025.csv
Rows: 1,860


...


========== SUMMARY STATISTICS ==========


,date,market_total_return (including dividends),risk_free_rate (1 month T-bill),market_excess_return,dividend_yield (annual),earnings_yield (annual),sp500_price,sp500_dividends (annual),sp500_earnings (annual)
count,1860,1200.0,1194.0,1194.0,1860.000000,1860.000000,1860.000000,1860.000000,1860.000000
mean,1948-07-15 17:01:09.677419264,0.009389,0.0027,0.006731,0.042217,0.072140,453.741220,8.552554,20.639446
min,1871-01-31 00:00:00,-0.291731,-0.0006,-0.292031,0.011087,0.008078,2.730000,0.180000,0.160000
25%,1909-10-23 06:00:00,-0.018227,0.0003,-0.020507,0.030035,0.053943,8.040000,0.430000,0.580000
50%,1948-07-15 12:00:00,0.013001,0.0023,0.010299,0.041983,0.066431,19.575000,0.970000,1.895000
75%,1987-04-07 12:00:00,0.039049,0.0042,0.036002,0.053509,0.086445,251.900000,8.355000,15.502500
max,2025-12-31 00:00:00,0.394144,0.0135,0.393144,0.138365,0.188235,6853.030000,79.520000,240.630000
std,NaN,0.052919,0.002489,0.053074,0.017637,0.027381,1025.212317,15.916544,41.851653



========== FULL SAMPLE SUMMARY ==========
First date overall: 1871-01-31
Last date overall: 2025-12-31

Months with all four variables: 1,194
Complete sample range: 1926-07-31 to 2025-12-31

========== VARIABLE DEFINITIONS ==========


,Variable,Definition
0,market_total_return (including dividends),Monthly CRSP value-weighted market total return
1,risk_free_rate (1 month T-bill),Monthly return on 1-month Treasury bills
2,market_excess_return,Monthly market return minus risk-free return
3,dividend_yield (annual),Annual dividends divided by S&P composite price
4,earnings_yield (annual),Annual earnings divided by S&P composite price
5,sp500_price,Shiller S&P composite monthly price
6,sp500_dividends (annual),Annual dividends per index unit
7,sp500_earnings (annual),Annual earnings per index unit



========== COMPLETE ==========
Output directory: /home/uwaterloo/nzanatta/A3
